In [1]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error , mean_squared_error , mean_absolute_percentage_error,r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("Air Traffic Data Cor Updated.csv",parse_dates=['Date'],index_col='Date')
df.head()

,domestic passengers,international passenegrs,domestic freight(in tonne),international freight(in tonne),GDP (in dollars),Jet Fuel Price per Gallon,Inflation Rate,Unemployement Rate
Date,,,,,,,,
2009-01-01,3288004,885435,20832,11675,1.341888e+12,71.75,10.88,7.66
2009-02-01,3293220,757168,18645,12482,1.341888e+12,61.97,10.88,7.66
2009-03-01,3122400,848046,23046,15359,1.341888e+12,65.01,10.88,7.66
2009-04-01,3266686,861715,21623,14512,1.341888e+12,68.55,10.88,7.66
2009-05-01,3883887,898410,19534,14586,1.341888e+12,72.22,10.88,7.66


In [3]:
df.index = pd.to_datetime(df.index)

Splitting into External Factors and Target Variables

In [4]:
# Endogenous variables (what we want to predict)
endog = df[['domestic passengers','international passenegrs','domestic freight(in tonne)','international freight(in tonne)']].astype(float)

# Exogenous variables (external factors)
exog = df[['GDP (in dollars)','Jet Fuel Price per Gallon','Inflation Rate ','Unemployement Rate']].astype(float)

Splitting the data into training and testing sets

In [5]:
train_size = int(len(df) * 0.8)
endog_train, endog_test = endog.iloc[:train_size], endog.iloc[train_size:]
exog_train, exog_test = exog.iloc[:train_size], exog.iloc[train_size:]

Fitting VARMAX model

In [6]:
model = VARMAX(endog_train, exog=exog_train, order=(2,1))
model_fit = model.fit(disp=False)
print(model_fit.summary())


                                                                            Statespace Model Results                                                                            
Dep. Variable:     ['domestic passengers', 'international passenegrs', 'domestic freight(in tonne)', 'international freight(in tonne)']   No. Observations:                  153
Model:                                                                                                                      VARMAX(2,1)   Log Likelihood               -7113.172
                                                                                                                            + intercept   AIC                          14382.344
Date:                                                                                                                  Wed, 12 Nov 2025   BIC                          14618.718
Time:                                                                                                              

Forecasting the next 12 months

In [7]:
forecast = model_fit.get_forecast(steps=len(endog_test), exog=exog_test)
forecast_df = forecast.predicted_mean

Evaluation Of the model

In [8]:
for col in endog.columns:
    mae = mean_absolute_error(endog_test[col], forecast_df[col])
    rmse= np.sqrt(mean_squared_error(endog_test[col], forecast_df[col]))
    mape=mean_absolute_percentage_error(endog_test[col], forecast_df[col])
    print(f"{col}: \nMAE={mae:.2f}\nRMSE={rmse:.2f}\nMAPE={mape:.2f}\n")


domestic passengers: 
MAE=2463131.73
RMSE=2639558.02
MAPE=0.20

international passenegrs: 
MAE=1253826.68
RMSE=1368496.74
MAPE=0.72

domestic freight(in tonne): 
MAE=2873.54
RMSE=3723.36
MAPE=0.05

international freight(in tonne): 
MAE=4890.46
RMSE=6303.91
MAPE=0.77



**Autoselection of the best parameters for p and q**

In [9]:
p_values = [1, 2, 3, 4, 5]   # autoregressive lags
q_values = [0, 1, 2, 3, 4]   # moving average lags

best_aic = float("inf")
best_order = None
best_model = None

for p in p_values:
    for q in q_values:
        try:
            model = VARMAX(endog_train, exog=exog_train, order=(p, q))
            model_fit = model.fit(disp=False)
            if model_fit.aic < best_aic:
                best_aic = model_fit.aic
                best_order = (p, q)
                best_model = model_fit
            print(f"Tested order=({p},{q}), AIC={model_fit.aic:.2f}")
        except Exception as e:
            print(f"Failed for order=({p},{q}) -> {e}")

Tested order=(1,0), AIC=14358.91
Tested order=(1,1), AIC=14369.30
Tested order=(1,2), AIC=14383.70
Tested order=(1,3), AIC=14408.66
Tested order=(1,4), AIC=14438.71
Tested order=(2,0), AIC=14351.49
Tested order=(2,1), AIC=14382.34
Tested order=(2,2), AIC=14400.66
Tested order=(2,3), AIC=14422.50
Tested order=(2,4), AIC=14444.74
Tested order=(3,0), AIC=14364.04
Tested order=(3,1), AIC=14394.81
Tested order=(3,2), AIC=14423.25
Tested order=(3,3), AIC=14454.17
Tested order=(3,4), AIC=14482.77
Tested order=(4,0), AIC=14373.34
Tested order=(4,1), AIC=14407.71
Tested order=(4,2), AIC=14437.50
Tested order=(4,3), AIC=14472.49
Tested order=(4,4), AIC=14502.47
Tested order=(5,0), AIC=14608.07
Tested order=(5,1), AIC=14652.25
Tested order=(5,2), AIC=14697.54
Tested order=(5,3), AIC=14751.38
Tested order=(5,4), AIC=14781.32


In [10]:
print(f"\nBest VARMAX order: {best_order} with AIC={best_aic:.2f}")



Best VARMAX order: (2, 0) with AIC=14351.49


In [11]:
forecast = best_model.get_forecast(steps=len(endog_test), exog=exog_test)
forecast_df = forecast.predicted_mean


In [12]:
for col in endog.columns:
    mae = mean_absolute_error(endog_test[col], forecast_df[col])
    rmse= np.sqrt(mean_squared_error(endog_test[col], forecast_df[col]))
    mape=mean_absolute_percentage_error(endog_test[col], forecast_df[col])
    r2 = r2_score(endog_test[col], forecast_df[col])
    print(f"{col}: \nMAE={mae:.2f}\nRMSE={rmse:.2f}\nMAPE={mape:.2f}\nR2 score={r2}")

domestic passengers: 
MAE=2462486.05
RMSE=2639190.49
MAPE=0.20
R2 score=-1.2715543130682008
international passenegrs: 
MAE=1253526.01
RMSE=1368405.33
MAPE=0.72
R2 score=-4.470813690057347
domestic freight(in tonne): 
MAE=2876.22
RMSE=3727.89
MAPE=0.05
R2 score=0.4897337524587275
international freight(in tonne): 
MAE=4891.57
RMSE=6305.14
MAPE=0.77
R2 score=-0.9979402749572392


**So for the values of the parameters p=2 and q=0 we get the least AIC = 14351.49**
